<a href="https://colab.research.google.com/github/dinazhuken/SASStudioWorkshop/blob/main/final_pipeline%2Bevaluation_code.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ==========================================
# PART 1: SETUP & INSTALLATION
# ==========================================
!apt-get update -y && apt-get install -y poppler-utils
!pip install -U torch torchvision torchaudio
# Notice the pillow<12.0 constraint added below!
!pip install -U "transformers>=4.45.0" "accelerate>=0.34.0" "einops" "sentencepiece" "huggingface_hub>=0.24.0" "timm" "pdf2image" "pillow<12.0" "qwen-vl-utils" -q

print("✅ Installation complete.")

import os
import gc
import re
import json
import torch
from pdf2image import convert_from_path
from PIL import Image
from transformers import AutoProcessor, AutoModelForImageTextToText, AutoTokenizer, AutoModelForCausalLM

# ==========================================
# CONFIGURATION
# ==========================================
drive.mount('/content/drive')

PDF_URL = "https://dn790004.ca.archive.org/0/items/dizionariolatino00calouoft/dizionariolatino00calouoft.pdf"
PDF_PATH = "/content/drive/My Drive/Thesis/dizionario.pdf"
OUTPUT_DIR = "/content/ocr_output"
PLAIN_TEXT_DIR = os.path.join(OUTPUT_DIR, "pages")

os.makedirs(PLAIN_TEXT_DIR, exist_ok=True)

if not os.path.exists(PDF_PATH):
    print(f"Downloading PDF...")
    !wget -q -O "$PDF_PATH" "$PDF_URL"


# The 5 sections you defined in Chapter 4 (A, D, I, M, R)
PAGE_RANGES = [
    (11, 15),
    (366, 371),
    (676, 681),
    (884, 889),
    (1158, 1162)
]

device = "cuda" if torch.cuda.is_available() else "cpu"

# ==========================================
# PART 2: VISION-LANGUAGE MODEL (OCR)
# ==========================================
print("\n" + "="*40)
print("🚀 STAGE A: VLM TRANSCRIPTION (A100 Optimized)")
print("="*40)

model_id_vlm = "JackChew/Qwen2-VL-2B-OCR"
print(f"Loading VLM model on {device} in bfloat16...")
processor = AutoProcessor.from_pretrained(model_id_vlm)
vlm_model = AutoModelForImageTextToText.from_pretrained(
    model_id_vlm,
    device_map="cuda",
    torch_dtype=torch.bfloat16 # A100 loves bfloat16
)

def get_image_slices(img, overlap=100):
    w, h = img.size
    slice_h = h // 3
    s1 = img.crop((0, 0, w, slice_h + overlap))
    s2 = img.crop((0, slice_h - overlap, w, (slice_h * 2) + overlap))
    s3 = img.crop((0, (slice_h * 2) - overlap, w, h))
    return [s1, s2, s3]

combined_text_paths = []

for start_page, end_page in PAGE_RANGES:
    print(f"\n📚 Converting PDF for pages {start_page} to {end_page}...")
    images = convert_from_path(PDF_PATH, first_page=start_page, last_page=end_page, fmt="png", dpi=300)

    combined_text_path = os.path.join(OUT_DIR, f"combined_ocr_{start_page}_{end_page}.txt")
    combined_text_paths.append(combined_text_path)
    full_text_buffer = []

    for i, page_image in enumerate(images):
        page_num = start_page + i
        print(f"--- Scanning Page {page_num} ---")
        slices = get_image_slices(page_image)
        page_parts = []

        for idx, img_slice in enumerate(slices):
            # Scale down slightly if massive, but A100 can handle large tensors
            if img_slice.height > 1500:
                 scale = 1500 / img_slice.height
                 new_w = int(img_slice.width * scale)
                 img_slice = img_slice.resize((new_w, 1500), Image.Resampling.LANCZOS)

            conversation = [{
                "role": "user",
                "content": [
                    {"type": "image"},
                    {"type": "text", "text": "Transcribe this dictionary text word for word. Do not describe the layout."}
                ]
            }]
            text_prompt = processor.apply_chat_template(conversation, add_generation_prompt=True)
            inputs = processor(text=[text_prompt], images=[img_slice], return_tensors="pt").to(device)

            with torch.inference_mode():
                output_ids = vlm_model.generate(**inputs, max_new_tokens=2048, do_sample=False)

            generated_ids = output_ids[:, inputs.input_ids.shape[-1]:]
            text_out = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]
            page_parts.append(text_out)

        full_text = "\n".join(page_parts)
        with open(os.path.join(plain_text_dir, f"page_{page_num}.txt"), "w", encoding="utf-8") as f:
            f.write(full_text)
        full_text_buffer.append(f"--- Page {page_num} ---\n{full_text}")

    with open(combined_text_path, "w", encoding="utf-8") as f:
        f.write("\n\n".join(full_text_buffer))
    print(f"✅ Section {start_page}-{end_page} saved to {combined_text_path}")

# CLEAR VRAM BEFORE LOADING THE NEXT MODEL
print("\n🧹 Unloading VLM to free GPU memory...")
del vlm_model
del processor
torch.cuda.empty_cache()
gc.collect()

# ==========================================
# PART 3: LLM EXTRACTION (JSON)
# ==========================================
print("\n" + "="*40)
print("🚀 STAGE B & C: SEGMENTATION & LLM EXTRACTION (A100 Optimized)")
print("="*40)

def clean_text(text):
    text = re.sub(r"--- Page \d+ ---", "", text)
    text = re.sub(r"\]*\]", "", text)
    text = re.sub(r"\(\d+,\d+\),\(\d+,\d+\)", "", text)
    text = re.sub(r"[\u4e00-\u9fff]+", "", text)
    text = text.replace("superficie, superficie, superficie", "")
    text = text.replace("\r", "")
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()

ENTRY_START_PATTERN = re.compile(
    r"^\s*(?P<hw>[a-zA-Z\u00C0-\u024F]+(?:[\s\-][a-zA-Z\u00C0-\u024F]+)*),\s+(?P<morph>.*)",
    re.MULTILINE
)

def segment_text_into_blocks(text):
    lines = text.split('\n')
    blocks, current_block = [], []
    for line in lines:
        line = line.strip()
        if not line: continue
        is_new_entry = ENTRY_START_PATTERN.match(line)
        if is_new_entry and len(line.split(',')[0]) < 35:
            if current_block: blocks.append("\n".join(current_block))
            current_block = [line]
        else:
            current_block.append(line)
    if current_block: blocks.append("\n".join(current_block))
    return blocks

MODEL_ID_LLM = "Qwen/Qwen2.5-7B-Instruct"
print(f"⏳ Loading {MODEL_ID_LLM} in pure bfloat16 (No Quantization)...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID_LLM)
# We load in pure bfloat16 instead of 4-bit because A100 is huge and fast!
llm_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID_LLM,
    device_map="cuda",
    torch_dtype=torch.bfloat16
)

def extract_json(block_text):
    prompt = f"""You are a Data Extraction Engine. Parse the dictionary entry below into a JSON object.
Input Entry: "{block_text}"
Rules:
1. "headword": The Latin word at the start (before the comma).
2. "morph": Grammar info immediately after the comma.
3. "def": The Italian definition. Merge lines. Do not translate. Do not summarize.
Output strictly valid JSON:
{{ "headword": "...", "morph": "...", "def": "..." }}"""

    messages = [{"role": "user", "content": prompt}]
    inputs = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(inputs, return_tensors="pt").to("cuda")

    with torch.inference_mode():
        outputs = llm_model.generate(**inputs, max_new_tokens=350, do_sample=False, temperature=0.0)

    response = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    try:
        start, end = response.find('{'), response.rfind('}') + 1
        if start != -1 and end != -1: return json.loads(response[start:end])
    except: pass
    return None

all_extracted_results = []

for text_path in combined_text_paths:
    print(f"\n📂 Processing {text_path}...")
    with open(text_path, "r", encoding="utf-8") as f:
        clean_txt = clean_text(f.read())
    blocks = segment_text_into_blocks(clean_txt)
    print(f"✂️ Found {len(blocks)} candidate entries in this section.")

    for i, block in enumerate(blocks):
        if len(block) < 10: continue
        entry = extract_json(block)
        if entry and entry.get("headword"):
            all_extracted_results.append(entry)

# ==========================================
# PART 4: POST-PROCESSING (SMART MERGE)
# ==========================================
print("\n" + "="*40)
print("🚀 STAGE D: POST-PROCESSING & MERGING")
print("="*40)

STRICT_STOPWORDS = {'il', 'lo', 'la', 'i', 'gli', 'le', 'un', 'uno', 'una', 'del', 'dello', 'della', 'dei', 'degli', 'delle', 'con', 'per', 'tra', 'fra', 'che', 'ed', 'ad', 'non', 'si', 'se'}

def is_garbage_headword(head):
    head_clean = head.strip()
    if not head_clean: return True
    if head_clean.lower() in STRICT_STOPWORDS: return True
    if head_clean[0].islower() and ' ' in head_clean: return True
    if len(head_clean) == 1 and head_clean.islower() and head_clean not in ['a', 'e']: return True
    return False

def smart_merge(entries):
    cleaned_entries, current_entry = [], None
    for entry in entries:
        head = entry.get('headword', '').strip()
        morph = entry.get('morph', '').strip()
        # Fallback in case LLM used 'definition' instead of 'def'
        defn = entry.get('def', entry.get('definition', '')).strip()

        if is_garbage_headword(head):
            if current_entry:
                current_entry['def'] += f" {head} {morph} {defn}".strip()
        else:
            if current_entry: cleaned_entries.append(current_entry)
            current_entry = {'headword': head, 'morph': morph, 'def': defn}

    if current_entry: cleaned_entries.append(current_entry)
    return cleaned_entries

print(f"Pre-merge Entry Count (All Sections): {len(all_extracted_results)}")
final_data = smart_merge(all_extracted_results)
print(f"Post-merge Final Count (All Sections): {len(final_data)}")

FINAL_OUT_PATH = os.path.join(OUT_DIR, "final_dictionary_full_dataset.json")
with open(FINAL_OUT_PATH, 'w', encoding='utf-8') as f:
    json.dump(final_data, f, indent=4, ensure_ascii=False)

print(f"🎉 SUCCESS! Full dataset saved to: {FINAL_OUT_PATH}")

Hit:1 https://cli.github.com/packages stable InRelease
Hit:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:5 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:10 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading packag

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/392 [00:00<?, ?it/s]

Qwen2VLForConditionalGeneration LOAD REPORT from: JackChew/Qwen2-VL-2B-OCR
Key                                                                     | Status     | 
------------------------------------------------------------------------+------------+-
base_model.model.visual.blocks.{0...31}.attn.proj.lora_B.default.weight | UNEXPECTED | 
base_model.model.visual.blocks.{0...31}.mlp.fc1.lora_B.default.weight   | UNEXPECTED | 
base_model.model.visual.blocks.{0...31}.attn.proj.lora_A.default.weight | UNEXPECTED | 
base_model.model.visual.blocks.{0...31}.attn.qkv.lora_B.default.weight  | UNEXPECTED | 
base_model.model.visual.blocks.{0...31}.attn.qkv.lora_A.default.weight  | UNEXPECTED | 
base_model.model.visual.blocks.{0...31}.mlp.fc2.lora_A.default.weight   | UNEXPECTED | 
base_model.model.visual.blocks.{0...31}.mlp.fc1.lora_A.default.weight   | UNEXPECTED | 
base_model.model.visual.blocks.{0...31}.mlp.fc2.lora_B.default.weight   | UNEXPECTED | 
model.visual.blocks.{0...31}.mlp.fc2.lora_B.d


📚 Converting PDF for pages 11 to 15...
--- Scanning Page 11 ---


...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]


📂 Processing /content/drive/My Drive/Thesis/ocr_output/combined_ocr_11_15.txt...
✂️ Found 115 candidate entries in this section.

📂 Processing /content/drive/My Drive/Thesis/ocr_output/combined_ocr_366_371.txt...
✂️ Found 76 candidate entries in this section.

📂 Processing /content/drive/My Drive/Thesis/ocr_output/combined_ocr_676_681.txt...
✂️ Found 201 candidate entries in this section.

📂 Processing /content/drive/My Drive/Thesis/ocr_output/combined_ocr_884_889.txt...
✂️ Found 299 candidate entries in this section.

📂 Processing /content/drive/My Drive/Thesis/ocr_output/combined_ocr_1158_1162.txt...
✂️ Found 70 candidate entries in this section.

🚀 STAGE D: POST-PROCESSING & MERGING
Pre-merge Entry Count (All Sections): 736
Post-merge Final Count (All Sections): 677
🎉 SUCCESS! Full dataset saved to: /content/drive/My Drive/Thesis/ocr_output/final_dictionary_full_dataset.json


In [ ]:
from google.colab import files
uploaded = files.upload()

Saving ground-truth1.json to ground-truth1.json


In [ ]:
!pip install nltk python-Levenshtein apted -q

import os
import json
import re
import unicodedata
import nltk
from Levenshtein import distance as levenshtein_distance

# Download NLTK tokenizers
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

# ==========================================
# 1. BULLETPROOF JSON LOADER
# ==========================================
def load_json_safely(filepath):
    with open(filepath, 'r', encoding='utf-8') as f:
        content = f.read()

    # Auto-fix 1: Fix any empty "def": fields (like the one in rĕ-cŏlo)
    content = re.sub(r'"def":\s*(?=\n\s*\})', r'"def": ""', content)

    # Auto-fix 2: Fix any missing commas between objects
    content = re.sub(r'\}\s*\{', '},\n{', content)

    # Auto-fix 3: Isolate ONLY the array and strip ALL hidden garbage at the end
    start = content.find('[')
    end = content.rfind(']')
    if start != -1 and end != -1:
        content = content[start:end+1]

    # Auto-fix 4: Remove any accidental commas right before the closing bracket
    content = re.sub(r',\s*\]$', '\n]', content)

    return json.loads(content)

# ==========================================
# 2. UTILITIES & NORMALIZATION
# ==========================================
def normalize_text(text, protocol="strict"):
    if not isinstance(text, str): return ""
    text = unicodedata.normalize('NFC', text.strip())
    if protocol == "soft":
        text = " ".join(text.lower().split())
    return text

def calculate_cer(ref, hyp):
    if len(ref) == 0: return 1.0 if len(hyp) > 0 else 0.0
    return levenshtein_distance(ref, hyp) / len(ref)

# ==========================================
# 3. ALIGNMENT PROTOCOL
# ==========================================
def align_entries(gold_json, pred_json, protocol="strict"):
    aligned_pairs = []
    unmatched_gold = list(gold_json)
    unmatched_pred = list(pred_json)

    for pred_item in unmatched_pred[:]:
        pred_hw = normalize_text(pred_item.get('headword', ''), protocol)
        best_match = None
        best_dist = float('inf')

        for gold_item in unmatched_gold:
            gold_hw = normalize_text(gold_item.get('headword', ''), protocol)
            dist = levenshtein_distance(pred_hw, gold_hw)

            if dist < best_dist:
                best_dist = dist
                best_match = gold_item

        if best_match is not None and best_dist <= max(len(pred_hw), len(gold_hw)) * 0.5:
            aligned_pairs.append((best_match, pred_item))
            unmatched_gold.remove(best_match)
            unmatched_pred.remove(pred_item)

    return aligned_pairs, unmatched_gold, unmatched_pred

# ==========================================
# 4. METRICS EVALUATION ENGINE
# ==========================================
def evaluate_pipeline(gold_json, pred_json, protocol="strict"):
    aligned_pairs, unmatched_gold, unmatched_pred = align_entries(gold_json, pred_json, protocol)

    tp = len(aligned_pairs)
    fp = len(unmatched_pred)
    fn = len(unmatched_gold)

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1_score = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

    total_hw_cer, total_def_cer = 0, 0
    kieval_tp = 0
    total_tokens_added = 0
    total_ned = 0

    for gold, pred in aligned_pairs:
        g_hw = normalize_text(gold.get('headword', ''), protocol)
        p_hw = normalize_text(pred.get('headword', ''), protocol)

        pred_def_raw = pred.get('def', pred.get('definition', ''))
        g_def = normalize_text(gold.get('def', ''), protocol)
        p_def = normalize_text(pred_def_raw, protocol)

        total_hw_cer += calculate_cer(g_hw, p_hw)
        total_def_cer += calculate_cer(g_def, p_def)

        max_len = max(len(g_def), len(p_def))
        sim = 1.0 - (levenshtein_distance(g_def, p_def) / max_len) if max_len > 0 else 1.0

        if protocol == "strict" and g_def == p_def:
            kieval_tp += 1
        elif protocol == "soft" and sim >= 0.70:
            kieval_tp += 1

        g_tokens = nltk.word_tokenize(g_def)
        p_tokens = nltk.word_tokenize(p_def)

        g_token_list = g_tokens.copy()
        tokens_added = 0
        for token in p_tokens:
            if token in g_token_list:
                g_token_list.remove(token)
            else:
                tokens_added += 1
        total_tokens_added += tokens_added
        total_ned += (1.0 - sim)

    avg_hw_cer = total_hw_cer / tp if tp > 0 else 0
    avg_def_cer = total_def_cer / tp if tp > 0 else 0

    kieval_p = kieval_tp / (kieval_tp + fp) if (kieval_tp + fp) > 0 else 0
    kieval_r = kieval_tp / (kieval_tp + fn) if (kieval_tp + fn) > 0 else 0
    kieval_f1 = (2 * kieval_p * kieval_r) / (kieval_p + kieval_r) if (kieval_p + kieval_r) > 0 else 0

    avg_tokens_added = total_tokens_added / max(len(pred_json), 1)
    avg_ned = total_ned / max(tp, 1)

    return {
        "Structural Precision": round(precision * 100, 2),
        "Structural Recall": round(recall * 100, 2),
        "Structural F1": round(f1_score * 100, 2),
        "Headword CER": round(avg_hw_cer, 4),
        "Definition CER": round(avg_def_cer, 4),
        "KIEval F1": round(kieval_f1 * 100, 2),
        "SCORE (Adj. NED)": round(avg_ned, 4),
        "TokensAdded (Avg)": round(avg_tokens_added, 2),
        "Total Aligned Entries": tp
    }

# ==========================================
# 5. EXECUTION LOOP
# ==========================================
if __name__ == "__main__":
    print("⏳ Loading and auto-cleaning datasets...")
    ground_truth = []

    # Load GT1 safely
    if os.path.exists('/content/ground-truth1.json'):
        try:
            ground_truth.extend(load_json_safely('/content/ground-truth1.json'))
            print("✅ Successfully loaded and cleaned ground-truth1.json!")
        except Exception as e:
            print(f"❌ Error in ground-truth1: {e}")

    # Load GT2 safely
    if os.path.exists('/content/ground-truth2.json'):
        try:
            ground_truth.extend(load_json_safely('/content/ground-truth2.json'))
            print("✅ Successfully loaded and cleaned ground-truth2.json!")
        except Exception as e:
            print(f"❌ Error in ground-truth2: {e}")

    # Load Predictions
    pred_path = '/content/drive/My Drive/Thesis/ocr_output/final_dictionary_full_dataset.json'
    try:
        # For predictions, we use the safe loader too just in case!
        predictions = load_json_safely(pred_path)
        print(f"✅ Successfully loaded predictions: {len(predictions)} entries found.")
    except Exception as e:
        print(f"❌ ERROR LOADING PREDICTIONS: {e}")
        predictions = []

    # Run Evaluation
    if len(ground_truth) > 0 and len(predictions) > 0:
        print(f"\nEvaluating {len(predictions)} predicted entries against {len(ground_truth)} gold entries...\n")

        print("="*40)
        print("📊 STRICT PROTOCOL (Exact Match)")
        print("="*40)
        strict_results = evaluate_pipeline(ground_truth, predictions, protocol="strict")
        for k, v in strict_results.items():
            print(f"{k}: {v}")

        print("\n" + "="*40)
        print("📊 SOFT PROTOCOL (Normalized)")
        print("="*40)
        soft_results = evaluate_pipeline(ground_truth, predictions, protocol="soft")
        for k, v in soft_results.items():
            print(f"{k}: {v}")
    else:
        print("\n❌ CRITICAL ERROR: Could not run evaluation.")

⏳ Loading and auto-cleaning datasets...
✅ Successfully loaded and cleaned ground-truth1.json!
✅ Successfully loaded predictions: 677 entries found.

Evaluating 677 predicted entries against 378 gold entries...

📊 STRICT PROTOCOL (Exact Match)
Structural Precision: 47.12
Structural Recall: 84.39
Structural F1: 60.47
Headword CER: 0.4767
Definition CER: 1.5452
KIEval F1: 1.88
SCORE (Adj. NED): 0.7452
TokensAdded (Avg): 8.64
Total Aligned Entries: 319

📊 SOFT PROTOCOL (Normalized)
Structural Precision: 47.56
Structural Recall: 85.19
Structural F1: 61.04
Headword CER: 0.4689
Definition CER: 1.4747
KIEval F1: 17.3
SCORE (Adj. NED): 0.736
TokensAdded (Avg): 8.6
Total Aligned Entries: 322


In [ ]:
import json
import re
import nltk
from Levenshtein import distance as levenshtein_distance

# Download NLTK tokenizers
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

# ==========================================
# 1. BULLETPROOF JSON LOADER
# ==========================================
def load_json_safely(filepath):
    with open(filepath, 'r', encoding='utf-8') as f:
        content = f.read()

    # Auto-fix 1: Fix any empty "def": fields (like the one in rĕ-cŏlo)
    content = re.sub(r'"def":\s*(?=\n\s*\})', r'"def": ""', content)

    # Auto-fix 2: Fix any missing commas between objects
    content = re.sub(r'\}\s*\{', '},\n{', content)

    # Auto-fix 3: Isolate ONLY the array and strip ALL hidden garbage at the end
    start = content.find('[')
    end = content.rfind(']')
    if start != -1 and end != -1:
        content = content[start:end+1]

    # Auto-fix 4: Remove any accidental commas right before the closing bracket
    content = re.sub(r',\s*\]$', '\n]', content)

    return json.loads(content)

# ==========================================
# 2. EXTRACT EXAMPLES FOR THESIS
# ==========================================
print("⏳ Loading data to find thesis examples...")

# Use the SAFE loader!
try:
    gold_data = load_json_safely('/content/ground-truth1.json')
    pred_data = load_json_safely('/content/drive/My Drive/Thesis/ocr_output/final_dictionary_full_dataset.json')
except Exception as e:
    print(f"Error loading files: {e}")
    gold_data = []
    pred_data = []

if gold_data and pred_data:
    # Basic alignment to separate matches from False Positives
    unmatched_pred = list(pred_data)
    aligned_pairs = []

    for gold in gold_data:
        g_hw = gold.get('headword', '').strip().lower()
        best_match = None
        best_dist = float('inf')

        for pred in unmatched_pred:
            p_hw = pred.get('headword', '').strip().lower()
            dist = levenshtein_distance(g_hw, p_hw)
            if dist < best_dist and dist <= max(len(g_hw), len(p_hw)) * 0.3:
                best_dist = dist
                best_match = pred

        if best_match:
            aligned_pairs.append((gold, best_match))
            unmatched_pred.remove(best_match)

    print("\n" + "="*60)
    print("📌 EXAMPLE 1: OVER-SEGMENTATION (FALSE POSITIVES)")
    print("These prove why your Precision is only 47%. The LLM read dense")
    print("dictionary columns and invented standalone entries.")
    print("="*60)
    for i, pred in enumerate(unmatched_pred[:3]):
        print(f"\n[Spurious Entry #{i+1}]")
        print(f"Headword: {pred.get('headword')}")
        print(f"Morph:    {pred.get('morph', '')}")
        print(f"Def:      {pred.get('def', pred.get('definition', ''))[:150]}...")

    print("\n" + "="*60)
    print("📌 EXAMPLE 2: FRAGMENTATION (LOW KIEval / HIGH CER)")
    print("These prove why KIEval is 17%. The LLM found the right headword,")
    print("but completely failed to attach the correct definition to it.")
    print("="*60)
    fragmented = []
    for gold, pred in aligned_pairs:
        g_def = gold.get('def', '')
        p_def = pred.get('def', pred.get('definition', ''))
        # Find examples where the definition lengths are drastically different
        if len(g_def) > 100 and len(p_def) < len(g_def) * 0.4:
            fragmented.append((gold, pred))

    for i, (gold, pred) in enumerate(fragmented[:2]):
        print(f"\n[Fragmented Entry #{i+1}] - Headword: {gold.get('headword')}")
        print(f"❌ LLM Extracted Def ({len(pred.get('def', pred.get('definition', '')))} chars): {pred.get('def', pred.get('definition', ''))}")
        print(f"✅ True Gold Def ({len(gold.get('def', ''))} chars):     {gold.get('def', '')[:200]}... [TRUNCATED]")

    print("\n" + "="*60)
    print("📌 EXAMPLE 3: HALLUCINATIONS (HIGH TokensAdded)")
    print("These prove that generative models invent text to 'fix' OCR noise.")
    print("="*60)
    hallucinations = []
    for gold, pred in aligned_pairs:
        g_def = gold.get('def', '').lower()
        p_def = pred.get('def', pred.get('definition', '')).lower()

        g_tokens = set(nltk.word_tokenize(g_def))
        p_tokens = set(nltk.word_tokenize(p_def))

        added_tokens = p_tokens - g_tokens
        # Find examples with lots of alphabetical added tokens
        added_words = [t for t in added_tokens if t.isalpha() and len(t) > 3]
        if len(added_words) > 5:
            hallucinations.append((gold, pred, added_words))

    for i, (gold, pred, words) in enumerate(hallucinations[:2]):
        print(f"\n[Hallucinated Entry #{i+1}] - Headword: {gold.get('headword')}")
        print(f"Tokens the LLM completely invented: {', '.join(words)}")
        print(f"LLM Def context: {pred.get('def', pred.get('definition', ''))[:150]}...")

⏳ Loading data to find thesis examples...

📌 EXAMPLE 1: OVER-SEGMENTATION (FALSE POSITIVES)
These prove why your Precision is only 47%. The LLM read dense
dictionary columns and invented standalone entries.

[Spurious Entry #1]
Headword: ábacus
Morph:    i, m. (ábax)
Def:      tavola divisa a mo' di scacchiera, originariamente con cifre (ABF), opp. asse della tavola coll'orlo rilevato, a) come tavoliere, tavola da giuoco, Su...

[Spurious Entry #2]
Headword: ábaliénatio
Morph:    ònis, f. (abalieno)
Def:      alienamento, espropriazione...

[Spurious Entry #3]
Headword: áb-álìeno
Morph:    ávi, átum, áre, alienare
Def:      I) propr., dar via, allontanare da sé, detto partic. di un possesso, espropriare, vendere, cedere (contr. conservare), Cic. II) trasl.: a) generic.: a...

📌 EXAMPLE 2: FRAGMENTATION (LOW KIEval / HIGH CER)
These prove why KIEval is 17%. The LLM found the right headword,
but completely failed to attach the correct definition to it.

[Fragmented Entry #1] - Headword: 

In [ ]:
import json
import nltk
from Levenshtein import distance as levenshtein_distance

def load_json_safely(filepath):
    import re
    with open(filepath, 'r', encoding='utf-8') as f: content = f.read()
    content = re.sub(r'"def":\s*(?=\n\s*\})', r'"def": ""', content)
    content = re.sub(r'\}\s*\{', '},\n{', content)
    start, end = content.find('['), content.rfind(']')
    if start != -1 and end != -1: content = content[start:end+1]
    content = re.sub(r',\s*\]$', '\n]', content)
    return json.loads(content)

print("⏳ Hunting for DENSE (D & R) Thesis Examples...")

gold_data = load_json_safely('/content/ground-truth1.json')
try:
    pred_data = load_json_safely('/content/drive/My Drive/Thesis/ocr_output/final_dictionary_full_dataset.json')
except:
    pred_data = []

if gold_data and pred_data:
    unmatched_pred = list(pred_data)
    aligned_pairs = []

    for gold in gold_data:
        g_hw = gold.get('headword', '').strip().lower()
        best_match = None
        best_dist = float('inf')
        for pred in unmatched_pred:
            p_hw = pred.get('headword', '').strip().lower()
            dist = levenshtein_distance(g_hw, p_hw)
            if dist < best_dist and dist <= max(len(g_hw), len(p_hw)) * 0.3:
                best_dist = dist
                best_match = pred
        if best_match:
            aligned_pairs.append((gold, best_match))
            unmatched_pred.remove(best_match)

    print("\n" + "="*60)
    print("📌 DENSE EXAMPLE 1: OVER-SEGMENTATION IN 'R' or 'D'")
    print("="*60)
    # Filter for R and D
    dense_spurious = [p for p in unmatched_pred if p.get('headword', '').lower().startswith(('r', 'd'))]
    for i, pred in enumerate(dense_spurious[:2]):
        print(f"\n[Dense Spurious Entry #{i+1}]")
        print(f"Headword: {pred.get('headword')}")
        print(f"Def:      {pred.get('def', pred.get('definition', ''))[:150]}...")

    print("\n" + "="*60)
    print("📌 DENSE EXAMPLE 2: FRAGMENTATION OF COMPLEX VERBS")
    print("="*60)
    dense_fragmented = []
    for gold, pred in aligned_pairs:
        g_hw = gold.get('headword', '').lower()
        if not g_hw.startswith(('r', 'd')): continue
        g_def = gold.get('def', '')
        p_def = pred.get('def', pred.get('definition', ''))
        # Look for massive gold entries that got brutally cut short
        if len(g_def) > 500 and len(p_def) < len(g_def) * 0.3:
            dense_fragmented.append((gold, pred))

    for i, (gold, pred) in enumerate(dense_fragmented[:2]):
        print(f"\n[Fragmented Verb #{i+1}] - Headword: {gold.get('headword')}")
        print(f"❌ LLM Extracted ({len(pred.get('def', ''))} chars): {pred.get('def', pred.get('definition', ''))[:150]}...")
        print(f"✅ True Gold Def ({len(gold.get('def', ''))} chars):     {gold.get('def', '')[:150]}...")

⏳ Hunting for DENSE (D & R) Thesis Examples...

📌 DENSE EXAMPLE 1: OVER-SEGMENTATION IN 'R' or 'D'

[Dense Spurious Entry #1]
Headword: demoralizzare
Def:      perdere...

[Dense Spurious Entry #2]
Headword: ritenere
Def:      così anche ritenere, sottrarre, non concedere,...

📌 DENSE EXAMPLE 2: FRAGMENTATION OF COMPLEX VERBS

[Fragmented Verb #1] - Headword: dē
❌ LLM Extracted (544 chars): da, via da (mentre ex significa)il detrarre il trar fuori dall'interno di q.c. «fuori da »). I) di spazio, 1) = da, via da, giù da, talora anche di, d...
✅ True Gold Def (5615 chars):     indicante distacco, allontanamento, da un oggetto, in cui q.c. si è trovato = da, via da (mentre ex significa il detrarre, il trar fuori dall'interno ...

[Fragmented Verb #2] - Headword: dē-cēdo
❌ LLM Extracted (811 chars): andarsene da, partire, allontanarsi, tirar via (contr. accedere, accostarsi, manere, restare), assol. oppure alla domanda «dōnde?» mediante avv. o de ...
✅ True Gold Def (3853 chars):     andar

In [ ]:
import json
import re
import unicodedata
import nltk
import numpy as np
from Levenshtein import distance as levenshtein_distance
from sklearn.utils import resample

# Download NLTK tokenizers
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

# ==========================================
# 1. UTILITIES & SAFE LOADING
# ==========================================
def load_json_safely(filepath):
    with open(filepath, 'r', encoding='utf-8') as f:
        content = f.read()
    content = re.sub(r'"def":\s*(?=\n\s*\})', r'"def": ""', content)
    content = re.sub(r'\}\s*\{', '},\n{', content)
    start, end = content.find('['), content.rfind(']')
    if start != -1 and end != -1: content = content[start:end+1]
    content = re.sub(r',\s*\]$', '\n]', content)
    return json.loads(content)

def normalize_text(text, protocol="soft"):
    if not isinstance(text, str): return ""
    text = unicodedata.normalize('NFC', text.strip())
    if protocol == "soft":
        text = " ".join(text.lower().split())
    return text

def calculate_cer(ref, hyp):
    if len(ref) == 0: return 1.0 if len(hyp) > 0 else 0.0
    return levenshtein_distance(ref, hyp) / len(ref)

# ==========================================
# 2. PER-ENTRY METRIC ENGINE
# ==========================================
def get_entry_metrics(gold, pred, protocol="soft"):
    """Calculates all metrics for a single aligned pair."""
    g_hw = normalize_text(gold.get('headword', ''), protocol)
    p_hw = normalize_text(pred.get('headword', ''), protocol)
    g_def = normalize_text(gold.get('def', ''), protocol)
    p_def = normalize_text(pred.get('def', pred.get('definition', '')), protocol)

    hw_cer = calculate_cer(g_hw, p_hw)
    def_cer = calculate_cer(g_def, p_def)

    max_len = max(len(g_def), len(p_def))
    sim = 1.0 - (levenshtein_distance(g_def, p_def) / max_len) if max_len > 0 else 1.0

    # KIEval: Correct grouping (sim >= 0.7 for soft)
    kieval_hit = 1 if sim >= 0.70 else 0

    # TokensAdded (Hallucination)
    g_tokens = nltk.word_tokenize(g_def)
    p_tokens = nltk.word_tokenize(p_def)
    g_token_list = g_tokens.copy()
    tokens_added = 0
    for token in p_tokens:
        if token in g_token_list: g_token_list.remove(token)
        else: tokens_added += 1

    return {
        "hw_cer": hw_cer,
        "def_cer": def_cer,
        "kieval_hit": kieval_hit,
        "ned": (1.0 - sim),
        "tokens_added": tokens_added
    }

# ==========================================
# 3. BOOTSTRAP EXECUTION
# ==========================================
def run_bootstrap_evaluation(gold_data, pred_data, n_iterations=1000):
    print(f"🚀 Running 95% Bootstrap ({n_iterations} iterations)...")

    # Align once to get the base populations
    unmatched_pred_global = list(pred_data)
    aligned_results = []

    for gold in gold_data:
        g_hw = normalize_text(gold.get('headword', ''), "soft")
        best_match = None
        best_dist = float('inf')
        for pred in unmatched_pred_global:
            p_hw = normalize_text(pred.get('headword', ''), "soft")
            dist = levenshtein_distance(g_hw, p_hw)
            if dist < best_dist and dist <= max(len(g_hw), len(p_hw)) * 0.5:
                best_dist = dist
                best_match = pred

        if best_match:
            metrics = get_entry_metrics(gold, best_match)
            aligned_results.append(metrics)
            unmatched_pred_global.remove(best_match)
        else:
            aligned_results.append(None) # False Negative

    # We also need to account for False Positives (unmatched predictions)
    total_fp = len(unmatched_pred_global)

    bootstrap_scores = {
        "Precision": [], "Recall": [], "F1": [],
        "Def CER": [], "KIEval F1": [], "TokensAdded": []
    }

    for i in range(n_iterations):
        # Sample gold entries with replacement
        sample = resample(aligned_results)

        tp = sum(1 for x in sample if x is not None)
        fn = sum(1 for x in sample if x is None)
        fp = total_fp # FP is constant for this estimation

        prec = tp / (tp + fp) if (tp + fp) > 0 else 0
        rec = tp / (tp + fn) if (tp + fn) > 0 else 0
        f1 = (2 * prec * rec) / (prec + rec) if (prec + rec) > 0 else 0

        # Mean of aligned metrics
        aligned_metrics = [x for x in sample if x is not None]
        avg_def_cer = np.mean([m['def_cer'] for m in aligned_metrics]) if tp > 0 else 0

        # KIEval
        kieval_tp = sum(m['kieval_hit'] for m in aligned_metrics)
        kp = kieval_tp / (kieval_tp + fp) if (kieval_tp + fp) > 0 else 0
        kr = kieval_tp / (kieval_tp + fn) if (kieval_tp + fn) > 0 else 0
        kf1 = (2 * kp * kr) / (kp + kr) if (kp + kr) > 0 else 0

        avg_tokens = np.mean([m['tokens_added'] for m in aligned_metrics]) if tp > 0 else 0

        bootstrap_scores["Precision"].append(prec * 100)
        bootstrap_scores["Recall"].append(rec * 100)
        bootstrap_scores["F1"].append(f1 * 100)
        bootstrap_scores["Def CER"].append(avg_def_cer)
        bootstrap_scores["KIEval F1"].append(kf1 * 100)
        bootstrap_scores["TokensAdded"].append(avg_tokens)

    print("\n" + "="*50)
    print("📊 TABLE 5.2 WITH 95% CONFIDENCE INTERVALS")
    print("="*50)
    for metric, values in bootstrap_scores.items():
        low = np.percentile(values, 2.5)
        high = np.percentile(values, 97.5)
        mean = np.mean(values)
        print(f"{metric:12}: {mean:6.2f} [{low:6.2f} - {high:6.2f}]")

# ==========================================
# 4. SUCCESS CASE FINDER
# ==========================================
def find_success_cases(gold_data, pred_data, count=2):
    print("\n" + "="*50)
    print("⭐ SUCCESS CASES (Chapter 5 Evidence)")
    print("="*50)

    successes = []
    unmatched_pred = list(pred_data)
    for gold in gold_data:
        g_hw = normalize_text(gold.get('headword', ''), "soft")
        for pred in unmatched_pred:
            p_hw = normalize_text(pred.get('headword', ''), "soft")
            if g_hw == p_hw:
                metrics = get_entry_metrics(gold, pred)
                # Success criteria: Very low CER and correct grouping
                if metrics['def_cer'] < 0.05 and metrics['kieval_hit'] == 1:
                    successes.append((gold, pred))
                    unmatched_pred.remove(pred)
                    break
        if len(successes) >= count: break

    for i, (g, p) in enumerate(successes):
        print(f"\n[Success Case #{i+1}]")
        print(f"Headword: {g.get('headword')}")
        print(f"Morph:    {p.get('morph', '')}")
        print(f"Full Def: {p.get('def', p.get('definition', ''))[:200]}...")

# ==========================================
# EXECUTION
# ==========================================
if __name__ == "__main__":
    gt_path = '/content/ground-truth1.json'
    pred_path = '/content/drive/My Drive/Thesis/ocr_output/final_dictionary_full_dataset.json'

    gold = load_json_safely(gt_path)
    pred = load_json_safely(pred_path)

    run_bootstrap_evaluation(gold, pred)
    find_success_cases(gold, pred)

🚀 Running 95% Bootstrap (1000 iterations)...

📊 TABLE 5.2 WITH 95% CONFIDENCE INTERVALS
Precision   :  40.35 [ 38.79 -  41.79]
Recall      :  72.31 [ 67.72 -  76.72]
F1          :  51.79 [ 49.33 -  54.10]
Def CER     :   1.29 [  0.94 -   1.69]
KIEval F1   :  18.63 [ 14.79 -  22.29]
TokensAdded :  15.31 [ 12.12 -  19.12]

⭐ SUCCESS CASES (Chapter 5 Evidence)

[Success Case #1]
Headword: dē-canto
Morph:    āvi, ātum, āre
Full Def: recitare cantando, I) tr., A) in gen.: elegos, Hor.: Halosin Illii, Suet. B) come term. spreg., cantar in musica, ricantare = ripetere fino alla sazietà, praeecepta, Cic.: II) intr., cessar dal canto ...

[Success Case #2]
Headword: Mylae
Morph:    ἀρυμ, f.
Full Def: castello di Zancle, posto sopra una lingua di terra ad occid. del Pelorum, nelle cui vicinanze Ottaviano sconfisse Sesto Pompeo in battaglia navale: oggi Milazzo....
